In [ ]:
import shap
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA    = Path.cwd().parent / 'data' / 'interim'
FIGURES = Path.cwd().parent / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

with open(DATA / 'best_model_rf.pkl', 'rb') as f:
    rf = pickle.load(f)

X_test  = pd.read_csv(DATA / 'X_test.csv')
y_test  = pd.read_csv(DATA / 'y_test.csv').squeeze()
results = pd.read_csv(DATA / 'test_predictions.csv')

print(f'RF model loaded  X_test: {X_test.shape}  churn: {y_test.mean()*100:.1f}%')


In [ ]:
# Compute SHAP values for the 90-day RF on the full test set
print('Computing SHAP values...')
explainer    = shap.TreeExplainer(rf)
shap_values  = explainer.shap_values(X_test)      # shape: (n, features, classes)
shap_churned = shap_values[:, :, 1]               # class 1 = churned
print(f'shap_churned shape: {shap_churned.shape}')

# Global importance ranking
mean_shap = pd.Series(np.abs(shap_churned).mean(axis=0), index=X_test.columns)
print('\nTop-10 features by mean |SHAP|:')
print(mean_shap.sort_values(ascending=False).head(10).round(4).to_string())


In [ ]:
# Global bar chart (thesis figure: shap_global_bar.png)
plt.figure(figsize=(9, 7))
shap.summary_plot(shap_churned, X_test, plot_type='bar', max_display=15, show=False)
plt.title('SHAP -- Global Feature Importance (Top 15)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIGURES / 'shap_global_bar.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Beeswarm (thesis figure: shap_beeswarm.png)
plt.figure(figsize=(9, 8))
shap.summary_plot(shap_churned, X_test, max_display=15, show=False)
plt.title('SHAP Beeswarm -- Feature Impact Direction', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(FIGURES / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Dependence plots for the two format-normalised features
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, feature in zip(axes, ['recency_normalized', 'visit_fulfillment']):
    shap.dependence_plot(feature, shap_churned, X_test, ax=ax, show=False)
    ax.set_title(f'SHAP Dependence -- {feature}', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / 'shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Waterfall for one churned client (thesis figure: shap_waterfall.png)
churned_idx = results[results['churned'] == 1].index[0]
exp = shap.Explanation(
    values        = shap_churned[churned_idx],
    base_values   = explainer.expected_value[1],
    data          = X_test.iloc[churned_idx].values,
    feature_names = X_test.columns.tolist()
)
plt.figure(figsize=(10, 7))
shap.waterfall_plot(exp, max_display=15, show=False)
plt.tight_layout()
plt.savefig(FIGURES / 'shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Profile verification table -- confirms numbers in thesis section 5.2.3
profiles_idx = {'C1': 8478, 'C2': 14047, 'C3': 1, 'R1': 1709, 'R2': 11, 'R3': 222}
rows = []
for code, idx in profiles_idx.items():
    proba = results.loc[idx, 'proba_rf']
    true  = 'churned' if results.loc[idx, 'churned'] == 1 else 'retained'
    row   = X_test.iloc[idx]
    sv    = pd.Series(shap_churned[idx], index=X_test.columns).sort_values(key=abs, ascending=False)
    rows.append({'Code': code, 'True': true, 'P(churn)': round(proba,3),
                 'tx_count': int(row['tx_count']), 'recency': round(row['recency'],1),
                 'rec_norm': round(row['recency_normalized'],2),
                 'vf': round(row['visit_fulfillment'],2),
                 'top_shap': f"{sv.index[0]} ({sv.iloc[0]:+.3f})"})
print(pd.DataFrame(rows).to_string(index=False))
